In [12]:
import requests
import numpy as np
import pandas as pd
import anndata as ad
import pooch
import os
import gdown

In [13]:
dataset_url = "https://datasets.cellxgene.cziscience.com/c55dc602-d168-4d15-acc1-5de4f2f5d551.h5ad"
 
# Google Drive share links -> file IDs (the long string between /d/ and /view)
model_url = "https://drive.google.com/file/d/1x1SfmFdI-zcocmqWAd7ZTC9CTEAVfKZq/view?usp=drive_link"
args_url = "https://drive.google.com/file/d/15TEZmd2cZCrHwgfE424fgQkGUZCXiYrR/view?usp=drive_link"
vocab_url = "https://drive.google.com/file/d/1jfT_T5n8WNbO9QZcLWObLdRG8lYFKH-Q/view?usp=drive_link"

h5ad_name = "adata_original.h5ad"

target_dir = "./data"
target_dir_scGPT = "./data/scGPT_CT"
os.makedirs(target_dir, exist_ok=True)
os.makedirs(target_dir_scGPT, exist_ok=True)

In [14]:
dataset_path = pooch.retrieve(
    url=dataset_url,
    fname=h5ad_name,
    path=target_dir,
    progressbar=True,
)
 
def gdown_if_missing(url: str, output: str) -> str:
    """Download `url` with gdown into `output`, unless it's already there."""
    if os.path.exists(output):
        print(f"Already downloaded, skipping: {output}")
        return output
    return gdown.download(
        url=url,
        output=output,
        quiet=False,
    )
 
 
# --- model files (Google Drive, via gdown) ---------------------------------
model_path = gdown_if_missing(model_url, os.path.join(target_dir_scGPT, "model.pt"))
print(f"Model available at: {model_path}")
 
args_path = gdown_if_missing(args_url, os.path.join(target_dir_scGPT, "args.json"))
print(f"Args available at: {args_path}")
 
vocab_path = gdown_if_missing(vocab_url, os.path.join(target_dir_scGPT, "vocab.json"))
print(f"Vocab available at: {vocab_path}")

Already downloaded, skipping: ./data/scGPT_CT/model.pt
Model available at: ./data/scGPT_CT/model.pt
Already downloaded, skipping: ./data/scGPT_CT/args.json
Args available at: ./data/scGPT_CT/args.json
Already downloaded, skipping: ./data/scGPT_CT/vocab.json
Vocab available at: ./data/scGPT_CT/vocab.json


In [15]:
print(f"Opening {dataset_path} in backed mode...")
adata_backed = ad.read_h5ad(dataset_path, backed="r+")
print(adata_backed)

#donor_id is Donor ID
#cell_type is cell type
#library_uuid is library run

Opening /Users/melinariepl/ramming_lab_code/data/adata_original.h5ad in backed mode...
AnnData object with n_obs × n_vars = 1263676 × 30172 backed at '/Users/melinariepl/ramming_lab_code/data/adata_original.h5ad'
    obs: 'library_uuid', 'assay_ontology_term_id', 'mapped_reference_annotation', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'cell_state', 'sample_uuid', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'disease_state', 'suspension_enriched_cell_types', 'suspension_uuid', 'suspension_type', 'donor_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'sex_ontology_term_id', 'Processing_Cohort', 'ct_cov', 'ind_cov', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'citation', 'default_embedding', 'is_pr

In [16]:
# 1. Determine cell subsample indices (operates purely on in-memory obs metadata)
donor_col = "donor_id"
n_per_donor = 200
rng = np.random.default_rng(0)

keep_idx = np.sort(np.concatenate([
    rng.choice(idx, size=min(n_per_donor, len(idx)), replace=False)
    for idx in adata_backed.obs.groupby(donor_col, observed=True).indices.values()
]))

# 2. Slice from adata.raw and load into memory
# .raw[keep_idx].to_adata() extracts raw.X into .X and keeps adata.obs metadata
adata_sub = adata_backed.raw[keep_idx].to_adata()

# 3. Rename var_names on the subsampled in-memory object
if "feature_name" in adata_sub.var.columns:
    adata_sub.var["ensembl_id"] = adata_sub.var_names.copy()
    adata_sub.var_names = adata_sub.var["feature_name"].astype(str)
    adata_sub.var_names.name = None
else:
    raise ValueError(
        "No 'feature_name' column in adata.raw.var — inspect adata_sub.var.columns."
    )

adata_sub.var_names_make_unique()

print(f"Subsampled: {adata_sub.n_obs} cells x {adata_sub.n_vars} genes, "
      f"{adata_sub.obs[donor_col].nunique()} donors")

# 4. Save the result
adata_sub.write_h5ad("./data/adata_subsampled.h5ad")

Subsampled: 52200 cells x 30172 genes, 261 donors


In [17]:
import scipy.sparse as sp

X = adata_sub.X

# 1. Identify data/matrix characteristics
is_sparse = sp.issparse(X)
values = X.data if is_sparse else X

# 2. Check min, max, and non-zero stats
min_val = values.min() if len(values) > 0 else 0
max_val = values.max() if len(values) > 0 else 0
has_negatives = min_val < 0

# 3. Check if values are integers
# Handles float types holding exact integers (e.g. 5.0) and native int types
if np.issubdtype(values.dtype, np.integer):
    is_integer = True
elif np.issubdtype(values.dtype, np.floating):
    # Sample up to 100k non-zero elements to check if they have non-zero decimal parts
    sample = values[:100_000]
    is_integer = np.all(np.mod(sample, 1) == 0)
else:
    is_integer = False

# 4. Print summary
print("--- Matrix Inspection ---")
print(f"Type:         {'Sparse (' + type(X).__name__ + ')' if is_sparse else 'Dense NumPy array'}")
print(f"Dtype:        {X.dtype}")
print(f"Value Range:  [{min_val}, {max_val}]")
print(f"Is Integer?   {is_integer}")
print(f"Has Negative? {has_negatives}")

if is_integer and not has_negatives:
    print(" Verdict:      Looks like raw count data (discrete, non-negative).")
elif not is_integer and not has_negatives:
    print(" Verdict:      Likely normalized, log-transformed, or scaled counts.")
elif has_negatives:
    print(" Verdict:      Likely z-score scaled or regressed data.")

--- Matrix Inspection ---
Type:         Sparse (csr_matrix)
Dtype:        float32
Value Range:  [1.0, 4576.0]
Is Integer?   True
Has Negative? False
 Verdict:      Looks like raw count data (discrete, non-negative).


In [18]:
# donor sanity check purely on adata_sub
donor_counts = adata_sub.obs[donor_col].value_counts()

print(f"\nDonors present: {adata_sub.obs[donor_col].nunique()}")
print(f"Cells/donor -> min: {donor_counts.min()}, median: {donor_counts.median():.0f}, "
      f"max: {donor_counts.max()} (target was {n_per_donor})")
print(f"Donors capped below target (<{n_per_donor} cells): "
      f"{(donor_counts < n_per_donor).sum()}")


Donors present: 261
Cells/donor -> min: 200, median: 200, max: 200 (target was 200)
Donors capped below target (<200 cells): 0
